# Can a Classifier with High Accuracy Still Be Practically Poor?
## EV Battery Fault / Health Classification

**Research question:** Can a classifier with high accuracy still be practically poor when the dataset is imbalanced?

This notebook uses the EV Lithium-Ion Battery Fault Diagnosis data. The **original dataset is balanced**. We intentionally create a controlled approximately 90:10 class imbalance for this experiment, so that we can see why accuracy alone can be misleading.

- Label `0`: Healthy battery
- Label `1`: Unhealthy battery


## 1. Import libraries
Only a Dummy Classifier and Logistic Regression are used in this project.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
CLASS_NAMES = ['Healthy', 'Unhealthy']


## 2. Upload and read the dataset
Upload the Excel file when the upload box appears. The code automatically finds the uploaded `.xlsx` file, so no filename is hardcoded.


In [ ]:
uploaded = files.upload()
excel_files = [name for name in uploaded if name.lower().endswith('.xlsx')]

if not excel_files:
    raise ValueError('Please upload an Excel file with the .xlsx extension.')

file_name = excel_files[0]
df = pd.read_excel(file_name)
print(f'Loaded file: {file_name}')


## 3. Dataset inspection
We inspect the data before modelling. This checks its structure and verifies the required feature and target columns.


In [ ]:
required_columns = ['SOC', 'Temperature', 'Voltage', 'Label']
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f'Required column(s) not found: {missing_columns}. Found: {df.columns.tolist()}')

# Labels must be 0 (Healthy) and 1 (Unhealthy).
df['Label'] = pd.to_numeric(df['Label'], errors='raise').astype(int)
if not set(df['Label'].unique()).issubset({0, 1}):
    raise ValueError('Label must contain only 0 (Healthy) and 1 (Unhealthy).')

print('Dataset shape:', df.shape)
print('Number of rows:', df.shape[0])
print('Number of columns:', df.shape[1])
print('Column names:', df.columns.tolist())
display(df.head())
print('Data types:')
display(df.dtypes)
print('Missing values by column:')
display(df.isnull().sum())
print('Duplicate rows:', df.duplicated().sum())
print('Descriptive statistics for numerical columns:')
display(df[['SOC', 'Temperature', 'Voltage']].describe())


## 4. Original class distribution
The original Mendeley dataset is balanced, with approximately equal Healthy and Unhealthy samples. We calculate the counts from the uploaded file instead of hardcoding them.


In [ ]:
def class_summary(data, heading):
    counts = data['Label'].value_counts().reindex([0, 1], fill_value=0)
    percentages = counts / len(data) * 100
    print(heading)
    print(f'Total samples: {len(data)}')
    print(f'Healthy (Label 0): {counts[0]} ({percentages[0]:.2f}%)')
    print(f'Unhealthy (Label 1): {counts[1]} ({percentages[1]:.2f}%)')
    return counts, percentages

def plot_distribution(counts, percentages, title, show_percentage=False):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    bars = ax.bar(CLASS_NAMES, counts.values, color=['#4C78A8', '#E45756'])
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Number of samples')
    for bar, count, pct in zip(bars, counts.values, percentages.values):
        label = f'{count}\n({pct:.1f}%)' if show_percentage else f'{count}'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), label,
                ha='center', va='bottom', fontweight='bold')
    ax.set_ylim(0, max(counts.values) * 1.18)
    plt.show()

original_counts, original_percentages = class_summary(df, 'Original dataset distribution')
plot_distribution(original_counts, original_percentages, 'Original Dataset Class Distribution')


## 5. Create the controlled imbalanced dataset
The original dataset is balanced. We intentionally create a controlled approximately 90:10 class imbalance for this experiment: all Healthy samples and 40 randomly selected Unhealthy samples, using `random_state=42`.


In [ ]:
healthy_df = df[df['Label'] == 0]
unhealthy_df = df[df['Label'] == 1]

if len(unhealthy_df) < 40:
    raise ValueError('The uploaded dataset has fewer than 40 Unhealthy samples.')

unhealthy_sample = unhealthy_df.sample(n=40, random_state=RANDOM_STATE)
imbalanced_df = pd.concat([healthy_df, unhealthy_sample])
imbalanced_df = imbalanced_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

imbalanced_counts, imbalanced_percentages = class_summary(
    imbalanced_df, 'Controlled imbalanced dataset distribution'
)
plot_distribution(imbalanced_counts, imbalanced_percentages,
                  'Controlled Imbalanced Dataset — Approximately 90:10',
                  show_percentage=True)
print('\nThe original dataset is balanced. We intentionally created a controlled approximately 90:10 class imbalance for this experiment.')


## 6. Features, target, and train/test split
`X` contains SOC, Temperature, and Voltage. `y` contains the target: 0 = Healthy and 1 = Unhealthy. `stratify=y` preserves approximately the same class proportion in both sets. The test set is held out and is never used for training.


In [ ]:
X = imbalanced_df[['SOC', 'Temperature', 'Voltage']]
y = imbalanced_df['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print('Training set size:', len(X_train))
print('Test set size:', len(X_test))
print('Training class counts:', y_train.value_counts().sort_index().to_dict())
print('Test class counts:', y_test.value_counts().sort_index().to_dict())


## 7. Metric meanings
- **Accuracy:** Out of all predictions, how many were correct?
- **Precision:** When the model predicts a battery as Unhealthy, how often is it actually Unhealthy?
- **Recall:** Out of all batteries that are actually Unhealthy, how many did the model successfully detect?
- **F1-score:** A combined measure of Precision and Recall.
- **Confusion matrix:** A table showing correct and incorrect predictions for each class.

Precision, recall, and F1-score below focus on Label 1 (Unhealthy), the minority class.


In [ ]:
def evaluate_model(model_name, y_true, predictions):
    metrics = {
        'Accuracy': accuracy_score(y_true, predictions),
        'Precision': precision_score(y_true, predictions, pos_label=1, zero_division=0),
        'Recall': recall_score(y_true, predictions, pos_label=1, zero_division=0),
        'F1-score': f1_score(y_true, predictions, pos_label=1, zero_division=0)
    }
    print(model_name)
    for metric, value in metrics.items():
        print(f'{metric}: {value * 100:.2f}%')
    print('\nClassification report:')
    print(classification_report(y_true, predictions, target_names=CLASS_NAMES, zero_division=0))
    return metrics

def plot_confusion_matrix(y_true, predictions, title):
    matrix = confusion_matrix(y_true, predictions, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.show()
    return matrix


## 8. Dummy Classifier baseline
The Dummy Classifier is not a real learning model. It does not learn relationships among SOC, Temperature, and Voltage. With `strategy='most_frequent'`, it always predicts the majority class, Healthy. Its purpose is to show how class imbalance can make accuracy misleading.


In [ ]:
dummy_model = DummyClassifier(strategy='most_frequent')
dummy_model.fit(X_train, y_train)
dummy_pred = dummy_model.predict(X_test)

dummy_metrics = evaluate_model('Dummy Classifier results', y_test, dummy_pred)
dummy_matrix = plot_confusion_matrix(y_test, dummy_pred, 'Dummy Classifier Confusion Matrix')

if dummy_metrics['Recall'] == 0:
    print('Interpretation: The Dummy Classifier predicts only Healthy. It can have high accuracy because Healthy is the majority class, but it detects zero Unhealthy batteries (zero recall).')


## 9. Logistic Regression
Logistic Regression is the actual learning model. A `StandardScaler` and Logistic Regression are placed in one pipeline, so scaling is fitted using training data only. Conceptually: SOC, Temperature, and Voltage → Logistic Regression → probability of Unhealthy → Healthy / Unhealthy.


In [ ]:
lr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

lr_metrics = evaluate_model('Logistic Regression results', y_test, lr_pred)
lr_matrix = plot_confusion_matrix(y_test, lr_pred, 'Logistic Regression Confusion Matrix')


## 10. Confusion matrix explanation
- **TN:** Healthy predicted as Healthy.
- **TP:** Unhealthy predicted as Unhealthy.
- **FP:** Healthy predicted as Unhealthy.
- **FN:** Unhealthy predicted as Healthy.

False negatives are especially important here: an FN is an actually Unhealthy battery incorrectly predicted as Healthy — a missed failure.


## 11. Results table and comparison graphs
All results below are calculated from predictions on the held-out test set.


In [ ]:
results_table = pd.DataFrame([
    {'Model': 'Dummy Classifier', **{key: value * 100 for key, value in dummy_metrics.items()}},
    {'Model': 'Logistic Regression', **{key: value * 100 for key, value in lr_metrics.items()}}
])
results_table[['Accuracy', 'Precision', 'Recall', 'F1-score']] = results_table[
    ['Accuracy', 'Precision', 'Recall', 'F1-score']
].round(2)
display(results_table)

def grouped_metric_chart(data, metrics, title):
    positions = np.arange(len(metrics))
    width = 0.35
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, (_, row) in enumerate(data.iterrows()):
        bars = ax.bar(positions + (i - 0.5) * width, [row[m] for m in metrics],
                      width, label=row['Model'])
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)
    ax.set_xticks(positions)
    ax.set_xticklabels(metrics)
    ax.set_ylabel('Score (%)')
    ax.set_ylim(0, 100)
    ax.set_title(title, fontweight='bold')
    ax.legend()
    plt.show()

grouped_metric_chart(results_table, ['Accuracy', 'Precision', 'Recall', 'F1-score'],
                     'Model Performance Comparison')
grouped_metric_chart(results_table, ['Precision', 'Recall', 'F1-score'],
                     'Minority Class (Unhealthy) Performance')


## 12. Why accuracy can be misleading
This explanation uses the actual controlled distribution and test results.


In [ ]:
majority_pct = imbalanced_percentages[0]
minority_pct = imbalanced_percentages[1]
print(f'The controlled dataset contains {majority_pct:.2f}% Healthy and {minority_pct:.2f}% Unhealthy batteries.')
print(f'A model that predicts Healthy for every battery can obtain roughly {majority_pct:.2f}% accuracy simply because Healthy is the majority class.')
print(f'On this test set, the Dummy Classifier achieved {dummy_metrics["Accuracy"]*100:.2f}% accuracy but {dummy_metrics["Recall"]*100:.2f}% recall for Unhealthy batteries.')
print('Therefore, high accuracy does not necessarily mean good minority-class detection.')


## 13. Automatic result interpretation and answer to the research question
The following text is generated from the real test-set metrics and confusion matrices; it does not assume that Logistic Regression is better.


In [ ]:
dummy_fn = dummy_matrix[1, 0]
lr_fn = lr_matrix[1, 0]

print('AUTOMATIC INTERPRETATION')
print(f'Dummy Classifier — accuracy: {dummy_metrics["Accuracy"]*100:.2f}%, precision: {dummy_metrics["Precision"]*100:.2f}%, recall: {dummy_metrics["Recall"]*100:.2f}%, F1-score: {dummy_metrics["F1-score"]*100:.2f}%.')
print(f'Logistic Regression — accuracy: {lr_metrics["Accuracy"]*100:.2f}%, precision: {lr_metrics["Precision"]*100:.2f}%, recall: {lr_metrics["Recall"]*100:.2f}%, F1-score: {lr_metrics["F1-score"]*100:.2f}%.')
print(f'Dummy Classifier false negatives: {dummy_fn}. Logistic Regression false negatives: {lr_fn}.')

if dummy_metrics['Accuracy'] >= 0.80 and dummy_metrics['Recall'] < 0.50:
    print('Answer: Yes. In this experiment, the majority-class baseline has high accuracy while poor recall reveals that it misses most or all Unhealthy batteries.')
else:
    print('Answer: Review both accuracy and minority-class recall/F1 together. Accuracy alone does not show how reliably Unhealthy batteries are detected.')

if lr_metrics['Recall'] > dummy_metrics['Recall']:
    print('Logistic Regression detects a larger share of Unhealthy batteries than the Dummy Classifier on this test set, because it uses SOC, Temperature, and Voltage.')
elif lr_metrics['Recall'] == dummy_metrics['Recall']:
    print('Logistic Regression has the same Unhealthy recall as the Dummy Classifier on this test set.')
else:
    print('Logistic Regression has lower Unhealthy recall than the Dummy Classifier on this test set; this result should be discussed rather than assuming it is better.')
